# 22 · Account segmentation (work package 11)

**Work package 11** of the [applications plan](../../research/implementation_plan_applications.md), section 11.
A few trader types from behaviour, and whether the recommender should treat them differently.

- **Features** (`src/scripts/segments.py`, past only): activity (trades, tokens, active days, account age), what the
  account trades (share of popular-head tokens, age of tokens at its first trade, share of trades in tokens younger
  than a day), how it trades (hold time from first buy to first sell, buy share, trades per active day), and what
  the model expected of it at the previous point (median log rank of its pick-ups, WP9).
- **Method:** standardized features, k-means and a Gaussian mixture for $k$ = 3 – 10 on Dev A; $k$ and the kind
  by split-half stability (rule in `project_direction.md`, 24 November).
- **Useful if:** segments are stable over time, differ in how well the recommender serves them, and
  **(success criterion, stated in the plan)** a fixed blend with a popularity weight $w$ chosen per segment on Dev A
  beats the single-weight blend ($w$ = 0.5, frozen) on Dev B, paired by point, with a 95% interval excluding zero.
  Only then is it reported on the test, once.

The features and each account's recall at every weight $w \in \{0, 0.1, …, 1\}$ were built by
`src/scripts/build_segments.py` (data/project/wp11/).

In [1]:
import sys

sys.path.append("../scripts")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import adjusted_rand_score, silhouette_score

import farms as fm
import plots
import protocol as pc
import segments as sg
import stats
import token_data as td

plots.style()
pd.set_option("display.float_format", "{:,.3f}".format)
WP11 = td.PROJECT_DIR / "wp11"


def load(part: str, kind: str) -> pd.DataFrame:
    """All points of a part: features or recall tables with columns point and user."""
    tables = []
    for t in pc.POINTS[part]:
        table = pd.read_parquet(WP11 / f"{kind}_{t:%Y%m%d}.parquet").reset_index()
        tables.append(table.assign(point=t))
    return pd.concat(tables, ignore_index=True)


features = {part: load(part, "features") for part in pc.POINTS}
recall = {part: load(part, "recall") for part in pc.POINTS}
WEIGHT_COLUMNS = [c for c in recall["dev_a"].columns if c not in ("user", "point")]
pd.DataFrame({part: {"points": f["point"].nunique(), "account-points": len(f), "accounts": f["user"].nunique(),
                     "prev_log_rank missing": f["prev_log_rank"].isna().mean()} for part, f in features.items()})  # fmt: skip

,dev_a,dev_b,test
points,10.000,11.000,14.000
account-points,"73,482.000","67,890.000","102,086.000"
accounts,"39,104.000","30,254.000","44,839.000"
prev_log_rank missing,0.750,0.687,0.688


**Check:** the recall at $w$ = 0.5 must equal the frozen fixed blend's per-account recall saved for WP10.

In [2]:
frozen_blend = pd.read_parquet(td.PROJECT_DIR / "wp10" / "per_user_fixed_blend_dev_b.parquet")
joined = recall["dev_b"].merge(frozen_blend[["user", "point", "recall"]], on=["user", "point"])
print(f"Dev B: {len(joined):,} account-points; largest difference {np.abs(joined['0.5'] - joined['recall']).max():.2e}")

Dev B: 67,890 account-points; largest difference 0.00e+00


## 1. The features on Dev A (medians and spread)

In [3]:
features["dev_a"][sg.FEATURES].describe(percentiles=[0.1, 0.5, 0.9]).T[["10%", "50%", "90%", "mean"]]

,10%,50%,90%,mean
trades_14d,1.000,7.000,96.000,69.754
tokens_14d,1.000,3.000,39.000,16.008
days_active_14d,1.000,3.000,13.000,5.106
account_age_days,2.512,13.658,25.979,13.772
head_share,0.000,0.208,1.000,0.346
token_age_days,0.043,5.705,18.354,7.490
young_share,0.000,0.000,0.857,0.192
hold_hours,3.963,336.000,336.000,243.109
buy_share,0.000,0.529,1.000,0.562
trades_per_day,1.000,2.000,9.000,6.697


## 2. Choosing $k$ and the kind on Dev A

Up to 50,000 Dev A account-points (seed 0). Stability: the adjusted Rand index between models fitted on two random
halves, over 5 splits (mean and standard error). Silhouette on 5,000 rows, for reference.

In [4]:
fit_rows = features["dev_a"].sample(n=min(50_000, len(features["dev_a"])), random_state=0)
rows = []
for kind in ("kmeans", "mixture"):
    for k in range(3, 11):
        values = [sg.split_stability(fit_rows, k, kind, seed) for seed in range(5)]
        model = sg.fit(fit_rows, k, kind)
        sample = fit_rows.sample(5000, random_state=1)
        z = model.scaler.transform(sg._matrix(sample, model.fill))
        rows.append({"kind": kind, "k": k, "stability": np.mean(values), "se": np.std(values, ddof=1) / np.sqrt(len(values)),
                     "silhouette": silhouette_score(z, sg.assign(model, sample))})  # fmt: skip
choice_grid = pd.DataFrame(rows)
choice_grid

,kind,k,stability,se,silhouette
0,kmeans,3,0.987,0.002,0.239
1,kmeans,4,0.984,0.002,0.212
2,kmeans,5,0.938,0.022,0.196
3,kmeans,6,0.895,0.060,0.200
4,kmeans,7,0.774,0.073,0.207
5,kmeans,8,0.937,0.009,0.217
6,kmeans,9,0.785,0.062,0.221
7,kmeans,10,0.724,0.073,0.218
8,mixture,3,0.853,0.125,0.168
9,mixture,4,0.460,0.116,0.099


In [5]:
picks = {}
for kind, part in choice_grid.groupby("kind"):
    best = part.loc[part["stability"].idxmax()]
    within = part[part["stability"] >= best["stability"] - best["se"]]
    picks[kind] = within.sort_values("k").iloc[0]
KIND = max(picks, key=lambda kind: (picks[kind]["stability"], kind == "kmeans"))
K = int(picks[KIND]["k"])
print({kind: (int(p["k"]), round(float(p["stability"]), 3)) for kind, p in picks.items()})
print(f"Chosen: {KIND}, k = {K} (stability {picks[KIND]['stability']:.3f})")
model = sg.fit(fit_rows, K, KIND)
for part in pc.POINTS:
    features[part]["segment"] = sg.assign(model, features[part])

{'kmeans': (3, 0.987), 'mixture': (3, 0.853)}
Chosen: kmeans, k = 3 (stability 0.987)


## 3. Profiles

Medians of every feature per segment on Dev A, the segment's share of account-points, and the share of its accounts
in WP10's farm groups (built on Dev A; a label for the profile, not a clustering feature).

In [6]:
farm_members = set(fm.groups(fm.coaction_edges(fm.first_trades(td.load_legs(), pc.POINTS["dev_a"][0], pc.DEV_B_START), 5, 5)).index)
dev_a = features["dev_a"]
profile = dev_a.groupby("segment")[sg.FEATURES].median()
profile.insert(0, "share", dev_a["segment"].value_counts(normalize=True).sort_index())
profile["farm_share"] = dev_a.assign(farm=dev_a["user"].isin(farm_members)).groupby("segment")["farm"].mean()
profile

,share,trades_14d,tokens_14d,days_active_14d,account_age_days,head_share,token_age_days,young_share,hold_hours,buy_share,trades_per_day,prev_log_rank,farm_share
segment,,,,,,,,,,,,,
0,0.241,68.000,28.000,12.000,20.554,0.200,7.310,0.000,336.000,0.500,6.143,5.130,0.130
1,0.238,12.000,4.000,4.000,13.343,0.091,0.134,0.704,18.283,0.571,2.750,7.442,0.160
2,0.522,3.000,2.000,2.000,9.466,0.333,9.202,0.000,336.000,0.556,1.500,4.934,0.014


## 4. Stable over time?

Each account's segment at its last Dev A point against its first Dev B point (adjusted Rand index; 0 is chance),
and the size of each segment on Dev A, Dev B and test.

In [7]:
last_a = features["dev_a"].sort_values("point").groupby("user")["segment"].last()
first_b = features["dev_b"].sort_values("point").groupby("user")["segment"].first()
both = pd.concat([last_a.rename("dev_a"), first_b.rename("dev_b")], axis=1, join="inner")
print(f"Accounts at both: {len(both):,}; adjusted Rand index {adjusted_rand_score(both['dev_a'], both['dev_b']):.3f}; "
      f"same segment {np.mean(both['dev_a'] == both['dev_b']):.1%}")  # fmt: skip
sizes = pd.DataFrame({part: f["segment"].value_counts(normalize=True).sort_index() for part, f in features.items()})
sizes["largest ratio"] = sizes.max(axis=1) / sizes.min(axis=1)
sizes

Accounts at both: 10,741; adjusted Rand index 0.588; same segment 85.3%


,dev_a,dev_b,test,largest ratio
segment,,,,
0,0.241,0.296,0.333,1.382
1,0.238,0.204,0.155,1.535
2,0.522,0.500,0.512,1.042


## 5. Does the recommender serve the segments differently?

Mean recall per segment on Dev B: the frozen blend ($w$ = 0.5), the baseline (per-account recall saved for WP10),
and the best weight for the segment in hindsight (for scale only).

In [8]:
baseline_b = pd.read_parquet(td.PROJECT_DIR / "wp10" / "per_user_baseline_dev_b.parquet")[["user", "point", "recall"]]
dev_b = recall["dev_b"].merge(features["dev_b"][["user", "point", "segment"]], on=["user", "point"])
dev_b = dev_b.merge(baseline_b.rename(columns={"recall": "baseline"}), on=["user", "point"])
by_segment = dev_b.groupby("segment").agg(account_points=("user", "size"), baseline=("baseline", "mean"), blend_w05=("0.5", "mean"))
by_segment["best_w_in_hindsight"] = dev_b.groupby("segment")[WEIGHT_COLUMNS].mean().idxmax(axis=1)
by_segment

,account_points,baseline,blend_w05,best_w_in_hindsight
segment,,,,
0,20100,0.056,0.102,0.4
1,13819,0.018,0.037,0.6
2,33971,0.146,0.207,0.6


## 6. One weight per segment (the success criterion)

On Dev A: for each segment, the weight with the highest mean over points of the segment's mean recall. On Dev B:
every account gets its segment's weight; per point, mean recall against the single frozen weight 0.5.

In [9]:
dev_a_recall = recall["dev_a"].merge(features["dev_a"][["user", "point", "segment"]], on=["user", "point"])
per_point_segment = dev_a_recall.groupby(["segment", "point"])[WEIGHT_COLUMNS].mean().groupby("segment").mean()
weight_of_segment = {}
for segment, row in per_point_segment.iterrows():
    best = row.max()
    tied = [w for w in WEIGHT_COLUMNS if row[w] == best]
    weight_of_segment[segment] = min(tied, key=lambda w: abs(float(w) - 0.5))
print("Weight per segment (chosen on Dev A):", {int(s): float(w) for s, w in weight_of_segment.items()})
per_point_segment.round(3)

Weight per segment (chosen on Dev A): {0: 0.3, 1: 0.3, 2: 0.5}


,0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0
segment,,,,,,,,,,,
0,0.139,0.142,0.149,0.150,0.149,0.143,0.124,0.096,0.062,0.046,0.033
1,0.101,0.103,0.107,0.109,0.100,0.101,0.096,0.085,0.058,0.036,0.032
2,0.208,0.218,0.234,0.253,0.256,0.262,0.254,0.240,0.197,0.122,0.109


In [10]:
def per_segment_result(part: str) -> pd.DataFrame:
    """stats.compare of the per-segment blend against the single weight 0.5 on a part."""
    table = recall[part].merge(features[part][["user", "point", "segment"]], on=["user", "point"])
    own = table.apply(lambda r: r[weight_of_segment[r["segment"]]], axis=1)
    table = table.assign(per_segment=own)
    results = {"single w = 0.5": table.groupby("point")["0.5"].mean().rename("recall").reset_index(),
               "w per segment": table.groupby("point")["per_segment"].mean().rename("recall").reset_index()}  # fmt: skip
    return stats.compare(results, baseline="single w = 0.5")


result_b = per_segment_result("dev_b")
row = result_b.set_index("model").loc["w per segment"]
MET = bool(row["mean_difference"] > 0 and row["interval_low"] > 0)
print(f"Dev B: per-segment - single {row['mean_difference']:+.4f} [{row['interval_low']:+.4f}, {row['interval_high']:+.4f}], "
      f"wins {row['wins']} -> criterion {'MET' if MET else 'NOT met'}")  # fmt: skip
result_b

Dev B: per-segment - single -0.0008 [-0.0017, +0.0002], wins 4 of 11 -> criterion NOT met


,model,points,mean,lowest,highest,mean_difference,interval_low,interval_high,wins,sign_test_p
0,single w = 0.5,11,0.142,0.122,0.156,NaN,NaN,NaN,NaN,NaN
1,w per segment,11,0.141,0.121,0.157,-0.001,-0.002,0.000,4 of 11,0.549


Only if the criterion is met on Dev B is the per-segment blend reported on the test (once).

In [11]:
if MET:
    display(per_segment_result("test"))
else:
    print("Criterion not met on Dev B: the per-segment blend is not scored on the test.")

Criterion not met on Dev B: the per-segment blend is not scored on the test.


## 7. What this shows

- **Three trader types, very stable.** k-means with $k$ = 3 splits the same way on any half of the accounts
  (adjusted Rand index 0.987); more segments or the Gaussian mixture are much less stable. Named by their profile
  on Dev A:
  - **occasional traders** (52%): a median of 3 trades and 2 tokens in 14 days, on 2 days; tokens that are about
    9 days old when they first trade them; a third of their tokens in the popular head; almost no farm members
    (1.4%);
  - **active traders** (24%): 68 trades and 28 tokens, active on 12 of 14 days, older accounts; 13% farm members;
  - **launch flippers** (24%): 70% of their trades are in tokens younger than a day, which they first trade at a
    median age of 3 hours and sell after 18 hours; the model expected their previous pick-ups least (median rank
    about 1,700, against 140-170 for the others); 16% farm members.
- **Stable over time.** Of 10,741 accounts seen at both, 85% keep their segment from their last Dev A point to
  their first Dev B point (adjusted Rand index 0.59); segment sizes change by at most a factor of 1.5 across Dev A,
  Dev B and test (launch flippers shrink from 24% to 16%).
- **Served very differently.** On Dev B the fixed blend reaches recall 0.207 for occasional traders, 0.102 for
  active traders and 0.037 for launch flippers (baseline: 0.146, 0.056, 0.018). The blend roughly doubles the
  baseline in every segment, but launch flippers are barely reachable: they pick up tokens that are hours old,
  which is the cold-start problem of the project.
- **The success criterion is not met.** On Dev A the best popularity weight was 0.3 for active traders and launch
  flippers and 0.5 for occasional traders, but the differences in recall between these weights were small (at
  most 0.012), and on Dev B the per-segment blend is not better: −0.0008 (interval −0.0017 to +0.0002), 4 of 11
  points. In hindsight, Dev B would even have preferred 0.6 for launch flippers. One weight serves all segments
  equally well, so the per-segment blend is not scored on the test.
- **What it means:** the segments describe who the recommender helps (occasional traders of established tokens)
  and who it cannot (launch flippers); improving recall for the latter needs cold-start lists, not another weight.